# Phase 7 — Stage A: dictionary seeding

Use a Colab T4 GPU runtime. This stage trains a small self-supervised patch autoencoder, clusters 100,000 16×16 patch embeddings into 4,096 unit-normalized atoms, and records initial coherence. Dataset files and download archives stay on Colab's temporary disk and are never copied to Drive. Only the latest resumable encoder/K-Means checkpoints, status, append-only training logs, and final dictionary/metrics are saved to Drive. If Colab terminates, rerun setup and data acquisition, then rerun the training cell; encoder training/K-Means resume from their latest Drive checkpoints. Colab's temporary disk is cleared after runtime termination, so the dataset download must be repeated in a new runtime. The notebook clones or fast-forward updates `master`.

Stage A defaults to DIV2K only so the problematic Flickr2K source is skipped. Flickr2K is optional if you provide a working source or a folder already available in the Colab runtime. DIV2K is provided for academic research only; accept its terms only if your use complies.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/data/phase7')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_a')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True).stdout[:1000])

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
# Dataset and download archive are stored only on Colab's temporary disk.
# After runtime termination, rerun this cell to fetch the data again.
DATASETS = ['div2k']  # Add 'flickr2k' only if a working source is available.
DOWNLOAD_PUBLIC_DATASETS = True
# DIV2K is restricted to academic research; confirm terms before True.
ACCEPT_DIV2K_TERMS = True  # User accepted academic-research terms in the supplied Colab run.
# Optional Flickr2K fallback if it is already present in this runtime.
DRIVE_ARCHIVES = {}  # e.g. {'flickr2k': '/content/drive/MyDrive/Flickr2K.tar'}
DRIVE_FILE_IDS = {}  # e.g. {'flickr2k': 'Google-Drive-file-id'}
SOURCE_URLS = {}
command = [sys.executable, str(REPO / 'scripts' / 'prepare_phase7_datasets.py'), '--root', str(DATA_ROOT)]
for dataset in DATASETS:
    command.extend(['--dataset', dataset])
if DOWNLOAD_PUBLIC_DATASETS:
    command.append('--download-public')
if ACCEPT_DIV2K_TERMS:
    command.append('--accept-div2k-license')
for dataset, path in DRIVE_ARCHIVES.items():
    command.extend(['--drive-archive', f'{dataset}={path}'])
for dataset, file_id in DRIVE_FILE_IDS.items():
    command.extend(['--drive-file-id', f'{dataset}={file_id}'])
for dataset, url in SOURCE_URLS.items():
    command.extend(['--source-url', f'{dataset}={url}'])
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
from scripts.prepare_phase7_datasets import resolve_clean_image_dir
IMAGE_DIRS = [resolve_clean_image_dir(name, DATA_ROOT) for name in DATASETS]
print('Stage A image folders:', IMAGE_DIRS)

command = [
    sys.executable, str(REPO / 'scripts' / 'init_dictionaries.py'),
    '--output-dir', str(RUN_DIR), '--patch-count', '100000',
    '--cluster-count', '4096', '--embedding-dim', '64',
    '--encoder-epochs', '3', '--device', 'cuda'
]
for directory in IMAGE_DIRS:
    command.extend(['--image-dir', str(directory)])
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('Persistent Stage A artifacts:', sorted(path.name for path in RUN_DIR.iterdir()))